# 09 · Train a tiny GPT end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/09_tiny_gpt.ipynb) &nbsp; 📖 Concept deck: [Putting it together: a tiny GPT](https://nilesh-salpe.github.io/visual-ai/pytorch/09-putting-it-together/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a nanoGPT-style character model built from the earlier pieces (embedding, pre-norm blocks with SDPA, tied output head), trained with AdamW and a warmup + cosine schedule on CPU, then sampled.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

import math
TEXT = ("the quick brown fox jumps over the lazy dog. " * 3 + "a stitch in time saves nine. " * 3 + "all that glitters is not gold. " * 3) * 8
chars = sorted(set(TEXT)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in TEXT]); n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
def get_batch(split, B=16, T=32, g=None):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - T - 1, (B,), generator=g)
    return torch.stack([d[i:i + T] for i in ix]), torch.stack([d[i + 1:i + T + 1] for i in ix])

class Attn(nn.Module):
    def __init__(self, C, H):
        super().__init__(); self.H = H; self.qkv = nn.Linear(C, 3 * C, bias=False); self.proj = nn.Linear(C, C, bias=False)
    def forward(self, x):
        B, T, C = x.shape; q, k, v = self.qkv(x).split(C, 2)
        q, k, v = (t.view(B, T, self.H, C // self.H).transpose(1, 2) for t in (q, k, v))
        return self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, C))
class Blk(nn.Module):
    def __init__(self, C, H):
        super().__init__(); self.n1, self.a, self.n2 = nn.RMSNorm(C), Attn(C, H), nn.RMSNorm(C)
        self.m = nn.Sequential(nn.Linear(C, 4 * C, bias=False), nn.GELU(), nn.Linear(4 * C, C, bias=False))
    def forward(self, x):
        x = x + self.a(self.n1(x)); return x + self.m(self.n2(x))

## Task 1: The model

Token embedding (V × C) + learned position embedding (T_max × C), `L` blocks (`Blk` above), a final `nn.RMSNorm`, and an output head **tied** to the token embedding (`logits = h @ wte.weight.T`). `forward(idx, targets=None)` returns `(logits, loss)` with cross-entropy loss when targets are given.
Initialise both embeddings with `nn.init.normal_(…, std=0.02)` as GPT-2 does: `nn.Embedding` defaults to std 1, which with a tied head makes the initial logits huge (loss ≈ 46 instead of ≈ ln V).

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, V, C=64, H=4, L=2, T_max=64):
        super().__init__()
        # TODO
        raise NotImplementedError
    def forward(self, idx, targets=None):
        # TODO
        raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_TinyGPT(cls):
    torch.manual_seed(0); m = cls(V)
    x, y = get_batch("train", g=torch.Generator().manual_seed(0))
    logits, loss = m(x, y)
    assert logits.shape == (16, 32, V)
    assert abs(loss.item() - math.log(V)) < 0.6, f"an untrained model's loss should be near ln V = {math.log(V):.2f}"
    n_tied = sum(p.numel() for p in m.parameters())
    assert n_tied == V * 64 + 64 * 64 + 2 * (12 * 64 * 64 + 2 * 64) + 64, f"expected {V * 64 + 64 * 64 + 2 * (12 * 64 * 64 + 128) + 64} parameters with a tied head, got {n_tied}"

check("TinyGPT", _test_TinyGPT, TinyGPT)

## Task 2: Train it

Train for `steps` steps with AdamW (lr 3e-3, weight decay 0.1 on 2-D weights only), linear warmup 20 steps then cosine to 10%, gradient clipping at 1.0. Return `(train_losses, val_loss)` where val_loss is the mean over 10 validation batches under `torch.no_grad()` and `model.eval()`.

In [ ]:
def fit(model, steps=300, seed=0):
    g = torch.Generator().manual_seed(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_fit(f):
    torch.manual_seed(0); M = globals().get("TinyGPT_solution") or TinyGPT
    tl, vl = f(M(V), steps=300)
    assert tl[-1] < 0.5 * tl[0], "training loss should drop by more than half"
    assert vl < 1.0, f"validation loss should fall below 1.0 nats/char on this repetitive text, got {vl:.2f}"

check("fit", _test_fit, fit)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The model
class TinyGPT_solution(nn.Module):
    def __init__(self, V, C=64, H=4, L=2, T_max=64):
        super().__init__()
        self.wte, self.wpe = nn.Embedding(V, C), nn.Embedding(T_max, C)
        self.blocks = nn.ModuleList(Blk(C, H) for _ in range(L)); self.nf = nn.RMSNorm(C)
        nn.init.normal_(self.wte.weight, std=0.02); nn.init.normal_(self.wpe.weight, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.wte(idx) + self.wpe(torch.arange(T))
        for b in self.blocks: x = b(x)
        logits = self.nf(x) @ self.wte.weight.T
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

check("TinyGPT (reference)", _test_TinyGPT, TinyGPT_solution)

In [ ]:
# @title Solution for Task 2: Train it
def fit_solution(model, steps=300, seed=0):
    g = torch.Generator().manual_seed(seed)
    decay = [p for p in model.parameters() if p.dim() >= 2]; other = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": other, "weight_decay": 0.0}], lr=3e-3, betas=(0.9, 0.95))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / 20 if s < 20 else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1, (s - 20) / (steps - 20)))))
    losses = []
    model.train()
    for _ in range(steps):
        x, y = get_batch("train", g=g); _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        vl = sum(model(*get_batch("val", g=g))[1].item() for _ in range(10)) / 10
    return losses, vl

check("fit (reference)", _test_fit, fit_solution)

---
## Experiments
**Sample from it.** Feed a prompt, take the last position's logits, divide by a temperature, sample, append, repeat.

In [ ]:
torch.manual_seed(0); model = TinyGPT_solution(V); tl, vl = fit_solution(model, 300)
print(f"train loss {tl[0]:.2f} → {tl[-1]:.2f}, val loss {vl:.3f} nats/char (uniform would be {math.log(V):.2f})")
idx = torch.tensor([[stoi[c] for c in "the "]]); g = torch.Generator().manual_seed(1)
with torch.no_grad():
    for _ in range(80):
        logits, _ = model(idx[:, -64:]); p = F.softmax(logits[0, -1] / 0.8, -1)
        idx = torch.cat([idx, torch.multinomial(p, 1, generator=g)[None]], 1)
print("".join(chars[i] for i in idx[0].tolist()))

---
## Takeaways
* A GPT is embeddings + a stack of pre-norm blocks + a final norm + an output head (often tied to the embedding).
* The training loop is the same as for any model: AdamW with decay on matrices, warmup + cosine, clipping, `zero_grad` each step.
* Scale this exact code up (more layers, data, GPUs with FSDP and `torch.compile`) and you have nanoGPT.